# TX Port Choke Check

This notebook finds **FEGE ports whose RxMaxSpeed is choked** (stuck flat, not going upward).

It writes the same Excel report you already reviewed:
`TX_Port_Choke_Check.xlsx`

## How to use (beginner)

1. Copy **two files** into this folder on your PC:
   - `TX_Port_Choke_Check.ipynb` (this notebook)
   - `tx_port_choke_engine.py` (helper — do not edit)
2. Put your KPI file in the **same folder**:
   `D:\KPI Monitoring\Transmission\FEGEPortChockCheck`
3. Open this notebook in Jupyter.
4. Click **Run All**.
5. The report is saved in the **same folder**.

Accepted source types: **CSV, XLS, XLSX, XLSB** (binary Excel).


## Step 1 — Install libraries

Run this cell **once**.  
Each library has its install command in the comment next to the name.


In [ ]:
# pandas     ->  pip install pandas          (read tables)
# numpy      ->  pip install numpy           (numbers / percentiles)
# openpyxl   ->  pip install openpyxl        (read .xlsx)
# xlrd       ->  pip install xlrd            (read old .xls)
# pyxlsb     ->  pip install pyxlsb          (read binary Excel .xlsb)
# xlsxwriter ->  pip install xlsxwriter      (write the report)

import sys  # built in — no install
import subprocess  # built in — no install

packages = [
    "pandas",
    "numpy",
    "openpyxl",
    "xlrd",
    "pyxlsb",
    "xlsxwriter",
]

# This line installs the packages using the same Python that Jupyter is using
subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])

print("Libraries are ready.")


## Step 2 — Import libraries and the helper

Comments next to each import show the install command again.


In [ ]:
from pathlib import Path  # built in — no install
import sys  # built in — no install

import pandas as pd   # pip install pandas
import numpy as np    # pip install numpy

# Make sure Python can see the helper file sitting next to this notebook
HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import tx_port_choke_engine as engine  # local file — no pip install

print("Helper loaded from:", Path(engine.__file__).resolve())


## Step 3 — Your folder and file

- `FOLDER` is where your KPI file lives, and where the report will be saved.
- Leave `SOURCE_NAME` empty to auto-pick the newest CSV / XLS / XLSX / XLSB.
- Or type the exact file name, for example `FEG_KPI_5G Site_DHK.xlsb`.


In [ ]:
# === CHANGE THIS if your folder is different ===
FOLDER = Path(r"D:\KPI Monitoring\Transmission\FEGEPortChockCheck")

# === CHANGE THIS only if you want one specific file ===
# Examples: "FEG_KPI_5G Site_DHK.xlsb"  or  "my_export.csv"
# Leave "" to auto-find the newest KPI file in the folder
SOURCE_NAME = ""

# Report name (saved in the same folder)
OUTPUT_NAME = "TX_Port_Choke_Check.xlsx"

# If the Windows folder is not on this PC, use the current folder instead
if not FOLDER.exists():
    FOLDER = Path.cwd()
    print("Windows folder not found. Using current folder:")
else:
    print("Using folder:")

print(FOLDER)

if SOURCE_NAME:
    print("You chose this source file:", SOURCE_NAME)
else:
    print("Source file will be auto-detected (CSV / XLS / XLSX / XLSB).")


## Step 4 — Find and read the source file

The helper looks at the file ending:

| Ending | Meaning | Library |
|---|---|---|
| `.csv` | text table | pandas |
| `.xlsx` / `.xlsm` | modern Excel | openpyxl |
| `.xls` | old Excel | xlrd |
| `.xlsb` | binary Excel | pyxlsb |


In [ ]:
# Find the KPI file in your folder
source_path = engine.find_source_file(FOLDER, SOURCE_NAME)
print("Source file:", source_path)
print("File type:", source_path.suffix)

# Read it (CSV, XLS, XLSX or XLSB)
raw = engine.load_kpi_file(source_path)

# Show a small preview so you can confirm columns look right
print("Rows:", len(raw))
print("Columns:", list(raw.columns))
raw.head()


## Step 5 — What the rule does (plain words)

The code looks at **the last 7 days**, every hour (busy and night).

A site is listed if:

1. **At least 3 days** have **at least 3 hours** sitting on the same cap, **or**
2. **The last day** has **at least 2 hours** on the cap.

Two cap shapes:

- **Crowded level** — the line sits on one value (DHAPT35 / DHAPT48).
- **Hard ceiling** — the line moves, but every peak hits the same roof and never goes above it.

Severity is **how many hours** sat on the cap:

- Severe ≥ 50%
- High ≥ 25%
- Moderate ≥ 10%
- Low < 10%

Tx Total BW is shown as a column only. It is **not** used to grade the site.


## Step 6 — Run the check on every site


In [ ]:
# analyse() does this for each site:
#   1. convert RxMaxSpeed from bit/s to Mbit/s
#   2. keep only the latest 7 days
#   3. look for a crowded level, then a hard ceiling
#   4. count hours / days on the cap
#   5. give a severity name
work, records = engine.analyse(raw)

print("Date window:", work["Date"].min().date(), "to", work["Date"].max().date())
print("Sites in file:", work["eNodeB Name"].nunique())
print("Issue sites:", len(records))

# Preview the list inside the notebook
preview = pd.DataFrame(
    [
        {
            "Site": r["site"],
            "Severity": r["severity"],
            "Cap shape": r["cap_type"],
            "Tx Total BW (Mbit/s)": round(r["bw"], 2),
            "Stuck Rx (Mbit/s)": round(r["center"], 2),
            "Hours on cap %": round(r["hours_pct"], 1),
            "Days on cap": f"{r['days_on_cap']}/{r['days_total']}",
            "Last day cap": engine.last_day_text(r),
        }
        for r in records
    ]
)
preview


## Step 7 — Write the Excel report

The report is saved in the **same folder** as your source file.


In [ ]:
output_path = FOLDER / OUTPUT_NAME

# This writes: Summary, Site List, HourlyChartOfIssueSites, Hourly KPI, Method
engine._write_workbook(output_path, source_path.name, work, records)
engine.print_summary(work, records)

print()
print("DONE. Open this file:")
print(output_path)


## One-cell shortcut

After the libraries are installed, you can run **only this cell** next time.


In [ ]:
FOLDER = Path(r"D:\KPI Monitoring\Transmission\FEGEPortChockCheck")
if not FOLDER.exists():
    FOLDER = Path.cwd()

work, records, output_path = engine.run_report(
    folder=FOLDER,
    source_name="",                       # or "FEG_KPI_5G Site_DHK.xlsb"
    output_name="TX_Port_Choke_Check.xlsx",
)

print("Report saved to:", output_path)
